# Pipeline Correctness Test — `run_batch` components

This notebook validates that the full pipeline used by `run_batch.py`
works correctly end-to-end, on either CPU or GPU:

| Step | Module                          | Validates                                            |
| ---- | ------------------------------- | ---------------------------------------------------- |
| 0    | `s2omics.p0_ndpi_conversion`    | NDPI→TIFF extraction and pixel size metadata         |
| 1    | `p1_histology_preprocess`       | Rescaled H&E images are written                      |
| 2    | `p2_superpixel_quality_control` | QC mask pickle saved with correct shape              |
| 3    | `p3_feature_extraction`         | Embedding count matches tile grid, dims look correct |

**Demo data used:** `demo/trial2/`  
**Edit the config cell below**, then run all cells top-to-bottom.


In [ ]:
import torch

# ── Configuration — edit these before running ──────────────────────────────
PREFIX           = '../demo/trial2/'
SAVE_FOLDER      = '../demo/trial2/S2Omics_output'

FOUNDATION_MODEL = 'uni'    # 'uni' | 'virchow' | 'gigapath'
CKPT_PATH        = '../checkpoints/uni/'
DOWN_SAMP_STEP   = 40       # 10 = ~1:100 downsampling; 1 = every superpixel
BATCH_SIZE       = 32
NUM_WORKERS      = 0        # 0 avoids fork issues in notebooks; increase on clusters

# ── Auto-detect device ─────────────────────────────────────────────────────
if torch.cuda.is_available():
    try:
        cap  = torch.cuda.get_device_capability(0)
        arch = f"sm_{cap[0]}{cap[1]}"
        if arch in set(torch.cuda.get_arch_list()):
            DEVICE = 'cuda:0'
            print(f"GPU detected: {torch.cuda.get_device_name(0)}  (arch {arch}) → using DEVICE='cuda:0'")
        else:
            DEVICE = 'cpu'
            print(f"[WARN] GPU arch {arch} not supported by this PyTorch build (supports: {sorted(torch.cuda.get_arch_list())}).")
            print("       Falling back to DEVICE='cpu'.")
    except Exception as exc:
        DEVICE = 'cpu'
        print(f"[WARN] Could not inspect GPU: {exc} — using DEVICE='cpu'.")
else:
    DEVICE = 'cpu'
    print("No CUDA GPU available — using DEVICE='cpu'.")

print(f"\nDevice          : {DEVICE}")
print(f"Foundation model: {FOUNDATION_MODEL}")
print(f"Checkpoint path : {CKPT_PATH}")
print(f"Prefix          : {PREFIX}")
print(f"Save folder     : {SAVE_FOLDER}")
print(f"Down-samp step  : {DOWN_SAMP_STEP}  (batch size={BATCH_SIZE}, workers={NUM_WORKERS})")


# Step 0: NDPI to Image Conversion

Convert whole-slide image (NDPI/SVS/etc.) to a raw TIFF with pixel size metadata.

**Note:** Demo data is pre-extracted; this cell now imports the shared conversion helper from `s2omics/p0_ndpi_conversion.py`.
For real NDPI files, use the imported function below or run `run_batch_ndpi_to_step3.py` directly.


In [ ]:
import sys
import glob
sys.path.append('..')

from s2omics.p0_ndpi_conversion import convert_ndpi_to_image

# ────────────────────────────────────────────────────────────────────────────
# For real NDPI files, uncomment and run:
# ────────────────────────────────────────────────────────────────────────────
ndpi_path = "../demo/**/*.ndpi"
ndpi_files = glob.glob(ndpi_path, recursive=True)
for ndpi_file in ndpi_files:
    convert_ndpi_to_image(ndpi_file, output_dir=PREFIX)


In [ ]:
import os
from PIL import Image
import numpy as np

# ── Step 0 validation ──────────────────────────────────────────────────────
# Check he-raw image exists in any supported format
he_raw_found = False
he_raw_path = None
for ext in ( '.tif', '.tiff', '.ome.tif', '.png', '.svs'):
    candidate = os.path.join(PREFIX, 'he-raw' + ext)
    if os.path.exists(candidate):
        he_raw_found = True
        he_raw_path = candidate
        break

assert he_raw_found, \
    f"[FAIL] Missing he-raw image (any of .tiff/.tif/.png/.svs) under '{PREFIX}'"

print(f"[PASS] he-raw image found: {os.path.basename(he_raw_path)}")

# Load and check image properties
try:
    if he_raw_path.lower().endswith(('.tif', '.tiff', '.ome.tif')):
        import tifffile
        img_array = np.asarray(tifffile.imread(he_raw_path))
    else:
        img_array = np.asarray(Image.open(he_raw_path))

    h, w = img_array.shape[:2]
    print(f"       Dimensions: {h:,} × {w:,} pixels")
except Exception as exc:
    raise AssertionError(f"[FAIL] Could not read he-raw image: {exc}") from exc

# Check pixel-size-raw.txt
pixel_size_path = os.path.join(PREFIX, 'pixel-size-raw.txt')
assert os.path.exists(pixel_size_path), \
    f"[FAIL] Missing pixel-size-raw.txt under '{PREFIX}'"

with open(pixel_size_path, 'r') as f:
    pixel_size_str = f.read().strip()

try:
    pixel_size = float(pixel_size_str)
    print(f"[PASS] pixel-size-raw.txt: {pixel_size:.6f} µm/pixel")
    assert pixel_size > 0, "[FAIL] Pixel size must be > 0"
except ValueError:
    if pixel_size_str.lower() == 'unknown':
        print(f"[WARN] pixel-size-raw.txt: 'Unknown' (metadata missing from slide)")
    else:
        raise AssertionError(f"[FAIL] Invalid pixel size value: {pixel_size_str}") from None

print("\nStep 0 PASSED ✓  (NDPI→TIFF conversion validated)")


# Step 1: Preprocess the H&E image

Make sure the physical size of each pixel is 0.5 micron


In [ ]:
import sys
sys.path.append('..')
from s2omics.p1_histology_preprocess import histology_preprocess

histology_preprocess(PREFIX, show_image=True)


In [ ]:
import os

# ── Step 1 validation ──────────────────────────────────────────────────────
for base in ['he-scaled', 'he']:
    found = any(
        os.path.exists(os.path.join(PREFIX, base + ext))
        for ext in ('.tiff', '.tif', '.png')
    )
    assert found, f"[FAIL] Missing preprocessed image: '{base}' under '{PREFIX}'"
    print(f"[PASS] {base} exists")

print("\nStep 1 PASSED ✓")


# Step 2: Quality control for all superpixels

Superpixels are 8 microns \* 8 microns square-shaped pseudo cells

We use our new QC package HistoSweep for this procedure


In [ ]:
from s2omics.p2_superpixel_quality_control import superpixel_quality_control

superpixel_quality_control(PREFIX, SAVE_FOLDER, show_image=True)


In [ ]:
import os, pickle
import numpy as np

# ── Step 2 validation ──────────────────────────────────────────────────────
pickle_folder = SAVE_FOLDER.rstrip('/') + '/pickle_files/'

for fname, expected_type in [
    ('shapes.pickle',              dict),
    ('qc_preserve_indicator.pickle', np.ndarray),
]:
    path = os.path.join(pickle_folder, fname)
    assert os.path.exists(path), f"[FAIL] Missing: {path}"
    with open(path, 'rb') as f:
        obj = pickle.load(f)
    assert isinstance(obj, expected_type), \
        f"[FAIL] {fname}: expected {expected_type.__name__}, got {type(obj).__name__}"
    detail = f" shape={obj.shape} dtype={obj.dtype}" if isinstance(obj, np.ndarray) else f" keys={list(obj.keys())}"
    print(f"[PASS] {fname}{detail}")

# Sanity: QC mask must be 2-D boolean
mask = pickle.load(open(os.path.join(pickle_folder, 'qc_preserve_indicator.pickle'), 'rb'))
assert mask.ndim == 2, f"[FAIL] QC mask should be 2-D, got ndim={mask.ndim}"
tissue_frac = mask.mean()
print(f"       Tissue fraction: {tissue_frac:.2%}  (expected > 0)")
assert tissue_frac > 0, "[FAIL] QC mask is all-False — something went wrong in HistoSweep."

print("\nStep 2 PASSED ✓")


# Step 3: Histology feature extraction


In [ ]:
from s2omics.p3_feature_extraction import histology_feature_extraction

histology_feature_extraction(
    PREFIX, SAVE_FOLDER,
    foundation_model=FOUNDATION_MODEL,
    ckpt_path=CKPT_PATH,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    down_samp_step=DOWN_SAMP_STEP,
    num_workers=NUM_WORKERS,
)


In [ ]:
import os, glob, pickle
import numpy as np

# ── Step 3 validation ──────────────────────────────────────────────────────
pickle_folder = SAVE_FOLDER.rstrip('/') + '/pickle_files/'

# num_patches
num_patches_path = os.path.join(pickle_folder, 'num_patches.pickle')
assert os.path.exists(num_patches_path), f"[FAIL] Missing: {num_patches_path}"
with open(num_patches_path, 'rb') as f:
    num_patches = pickle.load(f)
print(f"[PASS] num_patches: {num_patches}  (grid = {num_patches[0]} x {num_patches[1]})")

# embedding files
pattern = os.path.join(
    pickle_folder,
    f'{FOUNDATION_MODEL}_embeddings_downsamp_{DOWN_SAMP_STEP}_part_*.pickle'
)
emb_files = sorted(glob.glob(pattern))
assert len(emb_files) > 0, \
    f"[FAIL] No embedding files found matching: {pattern}"

total, expected_dim = 0, None
for path in emb_files:
    with open(path, 'rb') as f:
        emb = pickle.load(f)
    arr0 = np.array(emb[0])
    if expected_dim is None:
        expected_dim = arr0.shape
    assert arr0.shape == expected_dim, \
        f"[FAIL] Inconsistent embedding dim in {os.path.basename(path)}: got {arr0.shape}, expected {expected_dim}"
    print(f"[PASS] {os.path.basename(path)}: {len(emb)} embeddings,  dim={arr0.shape}")
    total += len(emb)

expected_total = int(num_patches[0]) * int(num_patches[1])
print(f"\nTotal embeddings : {total}")
print(f"Expected (grid)  : {expected_total}")
assert total == expected_total, \
    f"[FAIL] Embedding count mismatch: got {total}, expected {expected_total}"

print(f"\nStep 3 PASSED ✓  (device used: {DEVICE})")


---

## Optional: Steps 4 & 5 — Histology segmentation & cluster merging

Run these only after Steps 1–3 pass validation.

# Step 4: Histology segmentation


In [ ]:
from s2omics.single_section.p4_get_histology_segmentation import get_histology_segmentation

get_histology_segmentation(
    PREFIX, SAVE_FOLDER,
    foundation_model=FOUNDATION_MODEL,
    down_samp_step=DOWN_SAMP_STEP,
    clustering_method='kmeans',
    n_clusters=20,
)


# Step 5: Merge over-clusters


In [ ]:
from s2omics.single_section.p5_merge_over_clusters import merge_over_clusters

merge_over_clusters(PREFIX, SAVE_FOLDER, target_n_clusters=15)
